#### Download packages if in Google Colab
If running in Google Colab, the following cell installs librosa, noisereduce, and soundfile before the rest of the notebook runs.

In [3]:
# List of pip packages to install when running in Google Colab (Colab may not have these pre-installed)
colab_requirements = [
    "pip install librosa",      # Audio analysis library (load, resample, etc.)
    "pip install noisereduce",  # Main noise reduction package
    "pip install soundfile",    # Read/write audio files (wav, flac, etc.)
]

import sys, subprocess  # sys: check environment; subprocess: run shell commands

def run_subprocess_command(cmd):
    """Run a shell command and stream its output line by line."""
    # Popen: spawn process; split() turns "pip install x" into ["pip","install","x"]
    process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE)
    # Decode bytes to string and print each line (strip removes trailing newline)
    for line in process.stdout:
        print(line.decode().strip())

# Check if we're running inside Google Colab (colab injects its modules)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # In Colab: install required packages before running the rest of the notebook
    for i in colab_requirements:
        run_subprocess_command(i)

# Test noise reduction algorithm and view steps of algorithm
This notebook demonstrates: (1) stationary vs non-stationary noise reduction, (2) synthetic and real noise, (3) parallel processing, (4) multichannel support, and (5) partial/subset denoising.

In [ ]:
# IPython.display: embed Audio, Image, etc. in notebook output
import IPython
# scipy.io.wavfile: alternative audio loader (we use soundfile below)
from scipy.io import wavfile
# Main noise reduction API: reduce_noise(), etc.
import noisereduce as nr
# soundfile: read/write audio from URLs, files; supports multiple formats
import soundfile as sf
# Generate band-limited white noise for testing
from noisereduce.generate_noise import band_limited_noise
# Plotting
import matplotlib.pyplot as plt
# Fetch files from URLs
import urllib.request
# Numerical arrays
import numpy as np
# io.BytesIO: wrap raw bytes so soundfile can read from memory (no temp file)
import io
# Render matplotlib plots inline in the notebook (not in a popup)
%matplotlib inline

### Load data
Download the sample audio (fish vocalization) from the noisereduce repo.

In [ ]:
# URL of a sample audio file (fish vocalization) from the noisereduce repo
url = "https://raw.githubusercontent.com/timsainb/noisereduce/master/assets/fish.wav"
# Open URL and get HTTP response object
response = urllib.request.urlopen(url)
# Read raw bytes, wrap in BytesIO so soundfile can read; sf.read returns (samples, sample_rate)
data, rate = sf.read(io.BytesIO(response.read()))
# Keep reference to data (redundant here; sometimes used after overwriting in later cells)
data = data

In [ ]:
# Display an inline audio player so you can listen to the original clean audio
IPython.display.Audio(data=data, rate=rate)

In [ ]:
# Create figure: width=20 inches, height=3 inches (wide for waveform)
fig, ax = plt.subplots(figsize=(20,3))
# Plot waveform: x = sample index, y = amplitude
ax.plot(data)

### Add noise
Add band-limited white noise (2–12 kHz) to create a synthetic noisy signal for testing.

In [ ]:
# Length of noise segment to use (seconds); full noise is same length as data
noise_len = 2  # seconds
# band_limited_noise: white noise filtered to freq range 2–12 kHz; *10 boosts amplitude
noise = band_limited_noise(min_freq=2000, max_freq=12000, samples=len(data), samplerate=rate) * 10
# Take first 2 seconds of noise (rate*noise_len samples); used later for stationary reference
noise_clip = noise[:rate * noise_len]
# Mix clean signal with noise to create noisy audio for testing
audio_clip_band_limited = data + noise

In [ ]:
# Plot the noisy waveform (signal + band-limited noise)
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_band_limited)

In [ ]:
# Play the noisy audio to hear the degradation before noise reduction
IPython.display.Audio(data=audio_clip_band_limited, rate=rate)

### Stationary remove noise
Stationary algorithm: uses a fixed threshold (mean + k×std per frequency) estimated from the signal or noise reference. Best for constant noise (fan, hiss).

In [ ]:
# Stationary noise reduction: uses fixed threshold (mean + 1.5*std) per frequency; needs noise ref or uses signal itself
reduced_noise = nr.reduce_noise(
    y=audio_clip_band_limited,   # Noisy input
    sr=rate,                     # Sample rate
    n_std_thresh_stationary=1.5, # Threshold = mean + 1.5*std (higher = more aggressive)
    stationary=True,             # Use stationary (not non-stationary) algorithm
)

In [ ]:
# Plot the result after stationary noise reduction
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(reduced_noise)

In [ ]:
# Listen to the stationary-denoised audio
IPython.display.Audio(data=reduced_noise, rate=rate)

### Non-stationary noise reduction
Non-stationary algorithm: adapts the noise floor over time (smoothing magnitude spectrogram). No explicit noise reference needed. Best for time-varying noise (speech, traffic).

In [ ]:
# Non-stationary noise reduction: adapts noise floor over time (no explicit noise ref needed)
reduced_noise = nr.reduce_noise(
    y=audio_clip_band_limited,     # Noisy input
    sr=rate,                       # Sample rate
    thresh_n_mult_nonstationary=2, # Signal must exceed smoothed magnitude by 2x to be kept
    stationary=False,              # Use non-stationary (adaptive) algorithm
)

In [ ]:
# Plot the result after non-stationary noise reduction
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(reduced_noise)

## A more difficult example
Use real-world cafe noise (non-stationary) mixed with the fish audio for a harder test. 

In [ ]:
# Load cafe/ambient noise (real-world, non-stationary) for a harder test
url = "https://raw.githubusercontent.com/timsainb/noisereduce/master/assets/cafe_short.wav"
response = urllib.request.urlopen(url)
# noise_data: cafe audio; noise_rate: its sample rate (may differ from main data)
noise_data, noise_rate = sf.read(io.BytesIO(response.read()))

In [ ]:
# Plot the cafe noise waveform
fig, ax = plt.subplots(figsize=(20,4))
ax.plot(noise_data)

In [ ]:
# Listen to the cafe noise alone
IPython.display.Audio(data=noise_data, rate=noise_rate)

### Add noise to data
Mix cafe noise with fish audio at SNR=2 (signal 2× stronger than noise).

In [ ]:
# SNR = 2 means signal power is 2x noise power (noisy mix)
snr = 2  # signal-to-noise ratio
# Scale noise so that when added to data, we get desired SNR
noise_clip = noise_data / snr
# Mix clean fish audio with cafe noise (challenging: real-world, time-varying noise)
audio_clip_cafe = data + noise_clip

### Plot noisy data
Visualize and listen to the noisy cafe mix.

In [ ]:
# Plot and play the cafe-mixed noisy audio
fig, ax = plt.subplots(figsize=(20,4))
ax.plot(audio_clip_cafe)
IPython.display.Audio(data=audio_clip_cafe, rate=noise_rate)

### Stationary remove noise (with cafe noise reference)
With y_noise=noise_clip, the algorithm learns the cafe noise profile and applies a fixed threshold.

In [ ]:
# Stationary reduction WITH noise reference: y_noise=noise_clip gives a clear noise profile
reduced_noise = nr.reduce_noise(
    y=audio_clip_cafe,            # Noisy mix (signal + cafe)
    sr=rate,                      # Sample rate
    y_noise=noise_clip,           # Pure noise reference for threshold estimation
    n_std_thresh_stationary=1.5,  # mean + 1.5*std per frequency
    stationary=True,
)

In [ ]:
# Overlay: original noisy (bottom) and stationary-denoised (top) waveforms
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_cafe)   # Noisy
ax.plot(reduced_noise)     # Denoised (may overlap/overshadow)

In [ ]:
# Listen to stationary-denoised cafe mix
IPython.display.Audio(data=reduced_noise, rate=rate)

### Non-stationary noise reduction
Adaptive algorithm: no y_noise needed; estimates noise floor over time.

In [ ]:
# Non-stationary: no y_noise needed; adapts to time-varying cafe noise
reduced_noise = nr.reduce_noise(
    y=audio_clip_cafe,
    sr=rate,
    thresh_n_mult_nonstationary=2,  # Keep bins 2x above smoothed magnitude
    stationary=False,
)

In [ ]:
# Compare noisy vs non-stationary denoised; alpha=1 keeps both visible
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_cafe)         # Noisy
ax.plot(reduced_noise, alpha=1)  # Denoised
IPython.display.Audio(data=reduced_noise, rate=rate)

In [ ]:
# Play non-stationary denoised audio again (duplicate cell for convenience)
IPython.display.Audio(data=reduced_noise, rate=rate)

### Ensure that noise reduction does not cause distortion when prop_decrease == 0
When prop_decrease=0, the mask floor is 1 (no suppression), so output should equal input. Verifies the pipeline doesn't distort clean audio.

In [ ]:
# prop_decrease=0: no suppression (mask floor = 1); output should equal input (no distortion)
noise_reduced = nr.reduce_noise(y=data, sr=rate, prop_decrease=0, stationary=False)

In [ ]:
# Two subplots: (1) zoomed segment 3000–5000; (2) full waveform
fig, axs = plt.subplots(nrows=2, figsize=(20,6))
# Top: zoomed view—original and prop_decrease=0 result should overlap
axs[0].plot(data[3000:5000])
axs[0].plot(noise_reduced[3000:5000])
# Bottom: full signal—verifies no distortion when prop_decrease=0
axs[1].plot(data)
axs[1].plot(noise_reduced)

In [ ]:
# Same as cell above: prop_decrease=0 → no change (sanity check)
noise_reduced = nr.reduce_noise(y=data, sr=rate, prop_decrease=0, stationary=False)

In [ ]:
# Same comparison as above: verify prop_decrease=0 causes no distortion
fig, axs = plt.subplots(nrows=2, figsize=(20,6))
axs[0].plot(data[3000:5000])
axs[0].plot(noise_reduced[3000:5000])
axs[1].plot(data)
axs[1].plot(noise_reduced)

### Reduce noise over batches in parallel on long signal
Repeat audio 10× to simulate a long recording; use n_jobs=2 for parallel chunk processing (CPU path only).

In [ ]:
# Repeat fish audio 10x to simulate a long recording (tests chunked/parallel processing)
long_data = np.tile(data, 10)
# Duration in seconds: total samples / sample rate
len(long_data) / rate

In [ ]:
# Plot the long (repeated) waveform
fig, ax = plt.subplots(figsize=(20,4))
ax.plot(long_data)

In [ ]:
# Add band-limited noise to the long signal
noise = band_limited_noise(min_freq=2000, max_freq=12000, samples=len(long_data), samplerate=rate) * 10
audio_clip_band_limited = long_data + noise

In [ ]:
# Plot the long noisy signal
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_band_limited)

In [ ]:
# Non-stationary with n_jobs=2: process chunks in parallel (2 worker processes)
reduced_noise = nr.reduce_noise(
    y=audio_clip_band_limited,
    sr=rate,
    thresh_n_mult_nonstationary=2,
    stationary=False,
    n_jobs=2,  # Parallel workers (use_torch must be False for n_jobs > 1)
)

In [ ]:
# Compare long noisy signal vs non-stationary denoised (parallel)
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_band_limited)
ax.plot(reduced_noise)

In [ ]:
# Same long signal with STATIONARY reduction and parallel workers
reduced_noise = nr.reduce_noise(
    y=audio_clip_band_limited,
    sr=rate,
    thresh_n_mult_nonstationary=2,  # Ignored when stationary=True; n_std_thresh_stationary used
    stationary=True,
    n_jobs=2,
)

In [ ]:
# Compare long noisy vs stationary-denoised (parallel)
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_band_limited)
ax.plot(reduced_noise)

### Reduce noise on only a subset of a long clip
Use SpectralGateStationary.get_traces(start_frame, end_frame) to denoise only samples 10000–20000.

In [ ]:
# Import low-level class to access get_traces(start_frame, end_frame) for partial processing
from noisereduce.noisereduce import SpectralGateStationary

In [ ]:
# Create SpectralGateStationary instance (does not process yet; just config)
sg = SpectralGateStationary(
    y=data,                        # Full audio (used for framing; only subset processed)
    sr=rate,
    y_noise=None,                  # Use signal itself as noise estimate (no ref)
    prop_decrease=1.0,             # Full suppression where mask=0
    time_constant_s=2.0,           # For non-stationary (not used in stationary)
    freq_mask_smooth_hz=500,       # Mask smoothing in freq (Hz)
    time_mask_smooth_ms=50,        # Mask smoothing in time (ms)
    n_std_thresh_stationary=1.5,   # Threshold = mean + 1.5*std
    tmp_folder=None,               # Temp file for parallel memmap
    chunk_size=600000,             # Samples per chunk
    padding=30000,                 # Overlap at chunk borders
    n_fft=1024,
    win_length=None,               # Defaults to n_fft
    hop_length=None,               # Defaults to win_length//4
    clip_noise_stationary=True,
    use_tqdm=False,
    n_jobs=1,
)

In [ ]:
# Process only frames 10000–20000 (partial denoising of a long clip)
subset_noise_reduce = sg.get_traces(start_frame=10000, end_frame=20000)

In [ ]:
# Plot only the denoised subset (10000–20000 samples)
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(subset_noise_reduce)

## Multichannel noise
Demonstrate stereo (2-channel) support: each channel is denoised independently.

In [ ]:
# Create stereo by duplicating the mono cafe-mix: shape (2, n_samples)
audio_clip_cafe_2_channel = np.vstack([audio_clip_cafe, audio_clip_cafe])
audio_clip_cafe_2_channel.shape  # (2, n_samples)

In [ ]:
# Multichannel: reduce_noise processes each channel; y shape (n_channels, n_samples)
reduced_noise = nr.reduce_noise(
    y=audio_clip_cafe_2_channel,
    sr=rate,
    n_std_thresh_stationary=1.5,
    stationary=True,
)

In [ ]:
# Output shape: same as input (2, n_samples)
reduced_noise.shape

In [ ]:
# Two subplots: channel 0 (L) and channel 1 (R); each shows noisy + denoised overlay
fig, axs = plt.subplots(nrows=2, figsize=(20,5))
axs[0].plot(audio_clip_cafe_2_channel[0])  # Noisy L
axs[1].plot(audio_clip_cafe_2_channel[1])  # Noisy R
axs[0].plot(reduced_noise[0])              # Denoised L
axs[1].plot(reduced_noise[1])              # Denoised R

In [ ]:
# Play stereo denoised audio (both channels)
IPython.display.Audio(data=reduced_noise, rate=rate)

In [ ]:
# Non-stationary on cafe mix (mono) again
reduced_noise = nr.reduce_noise(
    y=audio_clip_cafe,
    sr=rate,
    thresh_n_mult_nonstationary=2,
    stationary=False,
)

In [ ]:
# Shape: (n_samples,) for mono input
reduced_noise.shape

In [ ]:
# Final comparison: mono cafe mix, noisy vs non-stationary denoised
fig, ax = plt.subplots(figsize=(20,3))
ax.plot(audio_clip_cafe)
ax.plot(reduced_noise, alpha=1)
IPython.display.Audio(data=reduced_noise, rate=rate)